In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from arch.univariate import ARX
from scipy.stats import norm

### Import data

In [2]:
# path = os.path.abspath('E:/RA/Geert/task1.py')
# dir_path = os.path.dirname(path)
# os.chdir(dir_path)
# excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')



# Notebook folder (works in Jupyter)
base_dir = Path.cwd()

excel_path = base_dir / "Aggregate_CPI_inflation_20230513.xls"
excel_file = pd.ExcelFile(excel_path)



sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

/var/folders/_w/ndh_5fmn3sl6dqws00y6zbzm0000gp/T/ipykernel_7405/424048925.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))


In [12]:
# Notebook folder (works in Jupyter)
base_dir = Path.cwd()

excel_path = base_dir / "Aggregate_CPI_inflation_20230513.xls"
excel_file = pd.ExcelFile(excel_path)
  
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

/var/folders/_w/ndh_5fmn3sl6dqws00y6zbzm0000gp/T/ipykernel_69764/2778384059.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))


In [9]:
#sample_data = data_quarter[data_quarter['Year']>1969]
sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

In [40]:
sample_data = pd.read_pickle('../Aggregate_CPI_inflation.pkl')

## Model Selection

In [4]:
MeanModel = {'0,1':['Forecasted inflation'],
             '1,1':['Inflation_lag_1','Forecasted inflation'],
            '2,1':['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation'],
             '2,2':['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ],
            }

In [5]:
def GarchFamilyResults(df):
    garch11 = arch_model(df, mean='Constant', vol='GARCH',p=1, q=1,dist='normal').fit(disp='off',cov_type='hac')
    garch21 = arch_model(df, mean='Constant', vol='GARCH',p=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    garch12 = arch_model(df, mean='Constant', vol='GARCH',p=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    garch22 = arch_model(df, mean='Constant', vol='GARCH',p=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    print('\nGARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGARCH(1,1): ',garch11.aic,'\t',garch11.bic,
          '\nGARCH(2,1): ',garch21.aic,'\t',garch21.bic,
          '\nGARCH(1,2): ',garch12.aic,'\t',garch12.bic,
          '\nGARCH(2,2): ',garch22.aic,'\t',garch22.bic,)

    egarch111 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=1, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch211 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=1, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch112 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch121 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch221 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch122 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch212 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch222 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    print('\nEGARCH(p,o,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nEGARCH(1,1,1): ',egarch111.aic,'\t',egarch111.bic,
          '\nEGARCH(2,1,1): ',egarch211.aic,'\t',egarch211.bic,
          '\nEGARCH(1,1,2): ',egarch112.aic,'\t',egarch112.bic,
          '\nEGARCH(1,2,1): ',egarch121.aic,'\t',egarch121.bic,
          '\nEGARCH(2,2,1): ',egarch221.aic,'\t',egarch221.bic,
          '\nEGARCH(2,1,2): ',egarch212.aic,'\t',egarch212.bic,
          '\nEGARCH(1,2,2): ',egarch122.aic,'\t',egarch122.bic,
          '\nEGARCH(2,2,2): ',egarch222.aic,'\t',egarch222.bic,  )
    
    gjr_garch11 = arch_model(df, mean='Constant', vol='GARCH',p=1, o=1,q=1,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch12 = arch_model(df, mean='Constant', vol='GARCH',p=1, o=1,q=2,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch21 = arch_model(df, mean='Constant', vol='GARCH',p=2, o=2,q=1,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch22 = arch_model(df, mean='Constant', vol='GARCH',p=2, o=2,q=2,dist='normal').fit(disp='off',cov_type='hac')
    print('\nGJR-GARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGJR-GARCH(1,1): ',gjr_garch11.aic,'\t',gjr_garch11.bic,
          '\nGJR-GARCH(1,2): ',gjr_garch12.aic,'\t',gjr_garch12.bic,
          '\nGJR-GARCH(2,1): ',gjr_garch21.aic,'\t',gjr_garch21.bic,
          '\nGJR-GARCH(2,2): ',gjr_garch22.aic,'\t',gjr_garch22.bic,)

When estimating standard errors of Garch coefficients, this package provides three choices:
1. Classic standard errors (also known as White's or QMLE standard errors)
2. Robust standard errors (also known as Bollerslev-Wooldridge robust covariance estimator, which provides robustness to misspecification)
3. HAC (Heteroskedasticity and Autocorrelation Consistent) standard errors, suitable for models with autocorrelated errors.

I use HAC this time.

### AIC & BIC of Garch models under different mean models

I report AIC and BIC of variance models. Should I include AIC & BIC of mean model? How should I calculate this? Are below equations correct?

Calculate AIC  by 
$$
AIC_{total} = AIC_{mean model} + AIC_{variance model}
$$
Calculate BIC  by 
$$
BIC_{total} = -2 \times ( log L(mean model) + log L(variance model) ) + log(num of observations) \times (num of params)
$$

## Mean Model

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [10]:
GarchFamilyResults(sample_data['Inflation shock'])


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  389.09864444593717 	 402.44879676474244 
GARCH(2,1):  373.8416214080963 	 390.5293118066029 
GARCH(1,2):  391.0986442637959 	 407.78633466230247 
GARCH(2,2):  375.84162093308487 	 395.8668494112928

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  370.05087895317524 	 386.7385693516818 
EGARCH(2,1,1):  372.02736130099595 	 392.05258977920386 
EGARCH(1,1,2):  372.0508785797307 	 392.0761070579386 
EGARCH(1,2,1):  362.525432044273 	 382.5506605224809 
EGARCH(2,2,1):  362.0639991801501 	 385.4267657380593 
EGARCH(2,1,2):  374.0273610845375 	 397.3901276424467 
EGARCH(1,2,2):  364.5254323052536 	 387.88819886316287 
EGARCH(2,2,2):  364.0640027688753 	 390.76430740648584

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  371.30488851682264 	 387.9925789153292 
GJR-GARCH(1,2):  373.3048876254318 	 393.3301161036397 
GJR-GARCH(2,1):  358.9579743132183 	 382.32074087112755 
GJR-GARCH(2,2):  360.9579742821367 	 387.65827891974726

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [32]:
X = sm.add_constant(sample_data[['Inflation_lag_1','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  386.90877710127523 	 400.2972072241451 
GARCH(2,1):  377.3928084409839 	 394.12834609457127 
GARCH(1,2):  388.9087771095715 	 405.64431476315883 
GARCH(2,2):  379.3928086018968 	 399.47545378620157

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  370.5435640245301 	 387.27910167811746 
EGARCH(2,1,1):  367.23552355963136 	 387.31816874393616 
EGARCH(1,1,2):  372.54515054592855 	 392.62779573023334 
EGARCH(1,2,1):  363.4878452118638 	 383.5704903961686 
EGARCH(2,2,1):  365.44449078878324 	 388.87424350380553 
EGARCH(2,1,2):  369.2355234645821 	 392.6652761796044 
EGARCH(1,2,2):  365.4878456216318 	 388.91759833665407 
EGARCH(2,2,2):  367.3343507204163 	 394.111210966156

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  372.015056033051 	 388.75059368663835 
GJR-GARCH(1,2):  374.0150556725567 	 394.0977008568615 
GJR-GARCH(2,1):  364.33581583796683 	 387.7655685529891 
GJR-GARCH(2,2):  366.33581555353953 	 393.11267579927

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [33]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  385.16923731370787 	 398.55766743657773 
GARCH(2,1):  376.165182578026 	 392.9007202316134 
GARCH(1,2):  387.16923707665546 	 403.9047747302428 
GARCH(2,2):  378.16518248158934 	 398.24782766589414

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  371.0210658891882 	 387.75660354277557 
EGARCH(2,1,1):  366.82918044253904 	 386.91182562684384 
EGARCH(1,1,2):  373.0284530071094 	 393.1110981914142 
EGARCH(1,2,1):  363.1494867454422 	 383.23213192974697 
EGARCH(2,2,1):  365.08248585200056 	 388.51223856702285 
EGARCH(2,1,2):  368.8291804160407 	 392.258933131063 
EGARCH(1,2,2):  365.14948668227396 	 388.57923939729625 
EGARCH(2,2,2):  367.08248533969163 	 393.85934558543136

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  373.1078667759307 	 389.84340442951805 
GJR-GARCH(1,2):  375.107867651315 	 395.1905128356198 
GJR-GARCH(2,1):  363.1750248784663 	 386.60477759348856 
GJR-GARCH(2,2):  365.1750242746243 	 391.9518845203

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [34]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  385.46115879945717 	 398.84958892232703 
GARCH(2,1):  376.70207991086687 	 393.43761756445423 
GARCH(1,2):  387.46115912623975 	 404.1966967798271 
GARCH(2,2):  378.70207980341996 	 398.78472498772476

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  370.3158690481183 	 387.0514067017057 
EGARCH(2,1,1):  366.0042273200402 	 386.08687250434497 
EGARCH(1,1,2):  372.3158705462822 	 392.398515730587 
EGARCH(1,2,1):  363.04363605123103 	 383.1262812355358 
EGARCH(2,2,1):  364.9099255888666 	 388.3396783038889 
EGARCH(2,1,2):  368.01264633405117 	 391.44239904907346 
EGARCH(1,2,2):  365.0436359025696 	 388.47338861759187 
EGARCH(2,2,2):  366.90992468215893 	 393.68678492789866

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  373.40190258980795 	 390.1374402433953 
GJR-GARCH(1,2):  375.40190277598526 	 395.48454796029006 
GJR-GARCH(2,1):  363.67209055202005 	 387.10184326704234 
GJR-GARCH(2,2):  365.67209013773106 	 392.44895

### Present MLE details of best garch model in each mean models

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [36]:
arch_model(sample_data['Inflation shock'], mean='Constant', vol='GARCH',p=2, o=2,q=1,dist='normal').fit(disp='off',cov_type='hac')


                   Constant Mean - GJR-GARCH Model Results                    
Dep. Variable:        Inflation shock   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -174.267
Distribution:                  Normal   AIC:                           362.533
Method:            Maximum Likelihood   BIC:                           385.963
                                        No. Observations:                  210
Date:                Mon, May 04 2026   Df Residuals:                      209
Time:                        15:10:32   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0803  3.727e-02      2.154  3.123e-02 [7.236e-0

In [6]:
arch_model(sample_data['Inflation shock'], mean='Constant', vol='GARCH',p=2, o=2,q=1,dist='normal').fit(disp='off',cov_type='hac')


                   Constant Mean - GJR-GARCH Model Results                    
Dep. Variable:        Inflation shock   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -174.267
Distribution:                  Normal   AIC:                           362.533
Method:            Maximum Likelihood   BIC:                           385.963
                                        No. Observations:                  210
Date:                Tue, May 05 2026   Df Residuals:                      209
Time:                        16:17:25   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0803  3.727e-02      2.154  3.123e-02 [7.236e-0

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [ ]:
X = sm.add_constant(sample_data[['Inflation_lag_1','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -175.744
Distribution:                  Normal   AIC:                           363.488
Method:            Maximum Likelihood   BIC:                           383.570
                                        No. Observations:                  210
Date:                Mon, May 04 2026   Df Residuals:                      209
Time:                        15:10:56   Df Model:                            1
                                  Mean Model                                 
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu            -0.0245  3.328e-02     -0.735      0.462 [-8.970e-02,4.076e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3427      0.132     -2.601  9.288e-03 [ -0.601,-8.449e-02]
alpha[1]       0.4801      0.143      3.352  8.027e-04    [  0.199,  0.761]
gamma[1]       0.0807      0.115      0.704      0.482    [ -0.144,  0.305]
gamma[2]       0.3314  9.904e-02      3.346  8.209e-04    [  0.137,  0.525]
beta[1]        0.6827      0.107      6.376  1.819e-10    [  0.473,  0.893]
===========================================================================

Covariance estimator: hac
"""

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [18]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -175.575
Distribution:                  Normal   AIC:                           363.149
Method:            Maximum Likelihood   BIC:                           383.232
                                        No. Observations:                  210
Date:                Mon, May 04 2026   Df Residuals:                      209
Time:                        15:00:44   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0466  3.433e-02     -1.359      0.174 [ -0.114,2.064e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3025      0.119     -2.536  1.120e-02 [ -0.536,-6.874e-02]
alpha[1]       0.5157      0.157      3.280  1.039e-03    [  0.208,  0.824]
gamma[1]       0.0136      0.116      0.117      0.907    [ -0.214,  0.241]
gamma[2]       0.3355  9.673e-02      3.468  5.239e-04    [  0.146,  0.525]
beta[1]        0.7278  9.648e-02      7.544  4.568e-14    [  0.539,  0.917]
===========================================================================

Covariance estimator: hac
"""

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [81]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -173.843
Distribution:                  Normal   AIC:                           359.685
Method:            Maximum Likelihood   BIC:                           379.710
                                        No. Observations:                  208
Date:                Wed, May 24 2023   Df Residuals:                      207
Time:                        17:06:35   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0479  3.484e-02     -1.376      0.169 [ -0.116,2.035e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3051      0.119     -2.560  1.046e-02 [ -0.539,-7.153e-02]
alpha[1]       0.5150      0.157      3.273  1.063e-03    [  0.207,  0.823]
gamma[1]   7.5207e-03      0.118  6.382e-02      0.949    [ -0.223,  0.238]
gamma[2]       0.3446  9.672e-02      3.563  3.670e-04    [  0.155,  0.534]
beta[1]        0.7265  9.556e-02      7.603  2.903e-14    [  0.539,  0.914]
===========================================================================

Covariance estimator: hac
"""

In [7]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
print(am.summary())

                     Constant Mean - EGARCH Model Results                     
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -175.522
Distribution:                  Normal   AIC:                           363.044
Method:            Maximum Likelihood   BIC:                           383.126
                                        No. Observations:                  210
Date:                Tue, May 05 2026   Df Residuals:                      209
Time:                        16:17:54   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0466  3.467e-02     -1.343      0.179 [ -0.115,

In [41]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')


                     Constant Mean - EGARCH Model Results                     
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -175.522
Distribution:                  Normal   AIC:                           363.044
Method:            Maximum Likelihood   BIC:                           383.126
                                        No. Observations:                  210
Date:                Mon, May 04 2026   Df Residuals:                      209
Time:                        15:13:42   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0466  3.467e-02     -1.343      0.179 [ -0.115,